In [3]:
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq

files = sorted(Path("results_5").glob("detected_N*.parquet"))

print(f"Found {len(files)} files")

Found 456 files


In [7]:
import pandas as pd

In [4]:
output_file = Path("results_5/combined_detected.parquet")

# Read schema from first file
first_file = pq.ParquetFile(files[0])
schema = first_file.schema_arrow

# Temporary output
tmp_file = Path("results_5/combined_detected.tmp.parquet")

writer = pq.ParquetWriter(
    tmp_file,
    schema=schema,
    compression="zstd"
)

total_rows = 0

try:
    for i, file in enumerate(files, start=1):

        print(f"[{i}/{len(files)}] {file.name}")

        parquet_file = pq.ParquetFile(file)

        for batch in parquet_file.iter_batches(
            batch_size=100_000
        ):
            table = pa.Table.from_batches([batch])

            if not table.schema.equals(schema):
                raise ValueError(
                    f"Schema mismatch in {file}"
                )

            writer.write_table(table)

            total_rows += batch.num_rows

finally:
    writer.close()

print(f"\nTotal rows written: {total_rows:,}")

[1/456] detected_N139849_455.parquet
[2/456] detected_N200000_000.parquet
[3/456] detected_N200000_001.parquet
[4/456] detected_N200000_002.parquet
[5/456] detected_N200000_003.parquet
[6/456] detected_N200000_004.parquet
[7/456] detected_N200000_005.parquet
[8/456] detected_N200000_006.parquet
[9/456] detected_N200000_007.parquet
[10/456] detected_N200000_008.parquet
[11/456] detected_N200000_009.parquet
[12/456] detected_N200000_010.parquet
[13/456] detected_N200000_011.parquet
[14/456] detected_N200000_012.parquet
[15/456] detected_N200000_013.parquet
[16/456] detected_N200000_014.parquet
[17/456] detected_N200000_015.parquet
[18/456] detected_N200000_016.parquet
[19/456] detected_N200000_017.parquet
[20/456] detected_N200000_018.parquet
[21/456] detected_N200000_019.parquet
[22/456] detected_N200000_020.parquet
[23/456] detected_N200000_021.parquet
[24/456] detected_N200000_022.parquet
[25/456] detected_N200000_023.parquet
[26/456] detected_N200000_024.parquet
[27/456] detected_N20

In [5]:
expected_rows = 91_139_849

if total_rows != expected_rows:
    raise RuntimeError(
        f"Expected {expected_rows:,} rows, "
        f"but wrote {total_rows:,}"
    )

tmp_file.replace(output_file)

print(f"Created: {output_file}")

Created: results_5/combined_detected.parquet


In [ ]:
combined = pd.read_parquet("results_5/combined_detected.parquet")